In [ ]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug' as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [ ]:
from src.demos.ekg.baml_client.types import ReviewedOpportunity
from src.utils.pydantic.kv_store import PydanticStore

KV_STORE_ID = "file"
KEY = "cnes-venus-tma"

opportunity = PydanticStore(kvstore_id=KV_STORE_ID, model=ReviewedOpportunity).load_object(KEY)
assert opportunity
opportunity_json = opportunity.model_dump_json(indent=2)

In [ ]:
import kuzu
import pandas as pd

db = kuzu.Database(":memory:")
conn = kuzu.Connection(db)


partners_dicts = [partner.model_dump() for partner in opportunity.partners]
df = pd.DataFrame(partners_dicts)

result = conn.execute("LOAD FROM df RETURN *;")
print(result.get_as_df())

In [ ]:
from typing import Any, Callable, Dict, List, Optional, Tuple

from pydantic import BaseModel

from src.demos.ekg.baml_client.types import (
    CompetitiveLandscape,
    Customer,
    FinancialMetrics,
    Opportunity,
    Partner,
    Person,
    RiskAnalysis,
    TechnicalApproach,
)


class NodeInfo(BaseModel):
    """Configuration for extracting nodes from a Pydantic model."""

    baml_class: type[BaseModel]
    key: str  # Primary key field name
    indexed: bool = False
    field_path: Optional[str] = None  # Path to field in model (e.g., 'opportunity.customer')
    is_list: bool = False  # True if the field contains a list of items
    key_generator: Optional[Callable[[Dict[str, Any], str], str]] = None  # Custom key generator
    deduplication_key: Optional[str] = None  # Field to use for deduplication


class RelationInfo(BaseModel):
    """Configuration for extracting relationships from a Pydantic model."""

    from_node: type[BaseModel]
    to_node: type[BaseModel]
    name: str
    from_field_path: Optional[str] = None  # Path to source object
    to_field_path: str  # Path to target object(s)
    from_key_field: Optional[str] = None  # Override source key field
    to_key_field: Optional[str] = None  # Override target key field
    attributes: dict[str, str] = {}


# Helper function to restart database connection cleanly
def restart_database():
    """Restart the database connection to clear all tables."""
    global db, conn
    db = kuzu.Database(":memory:")
    conn = kuzu.Connection(db)
    print("🔄 Database restarted - all tables cleared")


# Helper function to create synthetic keys for NULL primary keys
def create_synthetic_key(data: Dict[str, Any], base_name: str) -> str:
    """Generate a synthetic key when primary key is None or empty."""
    return f"{base_name}_{hash(str(sorted(data.items()))) % 10000}"


# Enhanced graph schema with field paths for generic extraction

nodes = [
    NodeInfo(baml_class=Opportunity, key="name", indexed=True, field_path="opportunity"),
    NodeInfo(baml_class=Customer, key="name", indexed=True, field_path="opportunity.customer"),
    NodeInfo(
        baml_class=Person,
        key="name",
        indexed=True,
        field_path="opportunity.customer.contacts",
        is_list=True,
        deduplication_key="name",
    ),
    NodeInfo(baml_class=Person, key="name", indexed=True, field_path="team", is_list=True, deduplication_key="name"),
    NodeInfo(baml_class=Partner, key="name", indexed=True, field_path="partners", is_list=True),
    NodeInfo(baml_class=RiskAnalysis, key="risk_description", indexed=False, field_path="risks", is_list=True),
    NodeInfo(
        baml_class=FinancialMetrics,
        key="tcv",
        indexed=False,
        field_path="financials",
        key_generator=lambda data, base: str(data.get("tcv", 0.0)),
    ),
    NodeInfo(
        baml_class=TechnicalApproach,
        key="technical_stack",
        indexed=False,
        field_path="tech_stack",
        key_generator=lambda data, base: data.get("technical_stack") or f"{base}_tech_stack",
    ),
    NodeInfo(
        baml_class=CompetitiveLandscape,
        key="competitive_position",
        indexed=False,
        field_path="competition",
        key_generator=lambda data, base: data.get("competitive_position") or f"{base}_competitive_position",
    ),
]

relations = [
    RelationInfo(
        from_node=Opportunity,
        to_node=Customer,
        name="HAS_CUSTOMER",
        from_field_path="opportunity",
        to_field_path="opportunity.customer",
    ),
    RelationInfo(
        from_node=Customer,
        to_node=Person,
        name="HAS_CONTACT",
        from_field_path="opportunity.customer",
        to_field_path="opportunity.customer.contacts",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=Person,
        name="HAS_TEAM_MEMBER",
        from_field_path="opportunity",
        to_field_path="team",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=Partner,
        name="HAS_PARTNER",
        from_field_path="opportunity",
        to_field_path="partners",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=RiskAnalysis,
        name="HAS_RISK",
        from_field_path="opportunity",
        to_field_path="risks",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=FinancialMetrics,
        name="HAS_FINANCIALS",
        from_field_path="opportunity",
        to_field_path="financials",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=TechnicalApproach,
        name="HAS_TECH_STACK",
        from_field_path="opportunity",
        to_field_path="tech_stack",
    ),
    RelationInfo(
        from_node=Opportunity,
        to_node=CompetitiveLandscape,
        name="HAS_COMPETITION",
        from_field_path="opportunity",
        to_field_path="competition",
    ),
]

In [ ]:
def create_schema(conn, nodes: list[NodeInfo], relations: list[RelationInfo]) -> None:
    """Create node and relationship tables in Kuzu database.

    Creates CREATE NODE TABLE and CREATE REL TABLE statements based on NodeInfo
    and RelationInfo configurations. Handles table drops and recreation for
    idempotency.
    """
    # Drop existing tables in reverse dependency order
    for relation in relations:
        table_name = relation.name
        try:
            conn.execute(f"DROP TABLE {table_name};")
        except Exception:
            pass  # Table might not exist

    # Drop node tables (avoid duplicates)
    dropped_tables = set()
    for node in nodes:
        table_name = node.baml_class.__name__
        if table_name not in dropped_tables:
            try:
                conn.execute(f"DROP TABLE {table_name};")
                dropped_tables.add(table_name)
            except Exception:
                pass  # Table might not exist

    # Create node tables (avoid duplicates by tracking created table names)
    created_tables = set()
    for node in nodes:
        table_name = node.baml_class.__name__

        # Skip if table already created
        if table_name in created_tables:
            continue

        key_field = node.key

        # Build field definitions based on model
        fields = []
        model_fields = node.baml_class.model_fields

        for field_name, field_info in model_fields.items():
            # Map Python types to Kuzu types
            if hasattr(field_info.annotation, "__origin__"):
                # Handle typing constructs like List, Optional
                if field_info.annotation.__origin__ is list:
                    fields.append(f"{field_name} STRING[]")
                else:
                    fields.append(f"{field_name} STRING")
            elif field_info.annotation == float:
                fields.append(f"{field_name} DOUBLE")
            elif field_info.annotation == int:
                fields.append(f"{field_name} INT64")
            else:
                fields.append(f"{field_name} STRING")

        fields_str = ", ".join(fields)
        create_sql = f"CREATE NODE TABLE {table_name}({fields_str}, PRIMARY KEY({key_field}))"
        print(f"Creating node table: {create_sql}")
        conn.execute(create_sql)
        created_tables.add(table_name)

        # Kuzu doesn't support CREATE INDEX ON syntax yet, skipping for now
        # if node.indexed:
        #     index_sql = f"CREATE INDEX ON {table_name}({key_field})"
        #     print(f"Creating index: {index_sql}")
        #     try:
        #         conn.execute(index_sql)
        #     except Exception as e:
        #         print(f"Index creation failed: {e}")

    # Create relationship tables
    for relation in relations:
        from_table = relation.from_node.__name__
        to_table = relation.to_node.__name__
        rel_name = relation.name

        create_rel_sql = f"CREATE REL TABLE {rel_name}(FROM {from_table} TO {to_table})"
        print(f"Creating relationship table: {create_rel_sql}")
        conn.execute(create_rel_sql)


def get_field_by_path(obj: Any, path: str) -> Any:
    """Get a field from object using dot-separated path."""
    try:
        current = obj
        for part in path.split("."):
            if hasattr(current, part):
                current = getattr(current, part)
            elif isinstance(current, dict) and part in current:
                current = current[part]
            else:
                return None
        return current
    except (AttributeError, KeyError, TypeError):
        return None


def extract_graph_data(
    model: BaseModel, nodes: list[NodeInfo], relations: list[RelationInfo]
) -> Tuple[Dict[str, List[Dict]], List[Tuple]]:
    """Generic extraction of nodes and relationships from any Pydantic model.

    Args:
        model: Any Pydantic model instance
        nodes: List of node extraction configurations
        relations: List of relationship extraction configurations

    Returns:
        nodes_dict: dict[node_type, list[node_properties]]
        relationships: list[(from_type, from_key, to_type, to_key, rel_name)]
    """
    nodes_dict = {}
    relationships = []
    node_registry = {}  # Track created nodes for deduplication

    # Initialize node collections
    for node_info in nodes:
        node_type = node_info.baml_class.__name__
        nodes_dict[node_type] = []
        node_registry[node_type] = set()

    # Extract nodes
    for node_info in nodes:
        node_type = node_info.baml_class.__name__

        # Get the field data using path
        field_data = get_field_by_path(model, node_info.field_path) if node_info.field_path else model

        if field_data is None:
            continue

        # Handle list vs single object
        items_to_process = field_data if node_info.is_list else [field_data]

        for item in items_to_process:
            if item is None:
                continue

            # Convert to dict if it's a Pydantic model
            if hasattr(item, "model_dump"):
                item_data = item.model_dump()
            elif isinstance(item, dict):
                item_data = item.copy()
            else:
                continue

            # Generate or get primary key
            if node_info.key_generator:
                key_value = node_info.key_generator(item_data, node_type)
            else:
                key_value = item_data.get(node_info.key)

            # Handle NULL keys
            if key_value is None or key_value == "":
                key_value = create_synthetic_key(item_data, node_type)
                item_data[node_info.key] = key_value

            # Check for deduplication
            dedup_key = node_info.deduplication_key or node_info.key
            dedup_value = item_data.get(dedup_key)

            if dedup_value and dedup_value not in node_registry[node_type]:
                nodes_dict[node_type].append(item_data)
                node_registry[node_type].add(dedup_value)
            elif not dedup_value:
                nodes_dict[node_type].append(item_data)

    # Extract relationships
    for relation_info in relations:
        from_type = relation_info.from_node.__name__
        to_type = relation_info.to_node.__name__

        # Get source objects
        from_data = get_field_by_path(model, relation_info.from_field_path) if relation_info.from_field_path else model
        to_data = get_field_by_path(model, relation_info.to_field_path)

        if from_data is None or to_data is None:
            continue

        # Find the corresponding node info for key fields
        from_node_info = next((n for n in nodes if n.baml_class.__name__ == from_type), None)
        to_node_info = next((n for n in nodes if n.baml_class.__name__ == to_type), None)

        if not from_node_info or not to_node_info:
            continue

        # Get from_key
        if hasattr(from_data, "model_dump"):
            from_dict = from_data.model_dump()
        else:
            from_dict = from_data

        from_key_field = relation_info.from_key_field or from_node_info.key
        from_key = from_dict.get(from_key_field)

        if from_node_info.key_generator and from_key is None:
            from_key = from_node_info.key_generator(from_dict, from_type)

        # Handle to_data (can be list)
        to_items = to_data if isinstance(to_data, list) else [to_data]

        for to_item in to_items:
            if to_item is None:
                continue

            if hasattr(to_item, "model_dump"):
                to_dict = to_item.model_dump()
            else:
                to_dict = to_item

            to_key_field = relation_info.to_key_field or to_node_info.key
            to_key = to_dict.get(to_key_field)

            if to_node_info.key_generator and to_key is None:
                to_key = to_node_info.key_generator(to_dict, to_type)

            if from_key and to_key:
                relationships.append((from_type, str(from_key), to_type, str(to_key), relation_info.name))

    return nodes_dict, relationships


def load_graph_data(conn, nodes_dict: Dict[str, List[Dict]], relationships: List[Tuple], nodes: list[NodeInfo]) -> None:
    """Load nodes and relationships into Kuzu database.

    Uses CREATE statements to insert data. For production use, consider
    using LOAD FROM for better performance with large datasets.
    """
    # Load nodes
    for node_type, node_list in nodes_dict.items():
        if not node_list:
            continue

        print(f"Loading {len(node_list)} {node_type} nodes...")

        for node_data in node_list:
            # Convert None values and escape strings
            cleaned_data = {}
            for key, value in node_data.items():
                if value is None:
                    cleaned_data[key] = "NULL"
                elif isinstance(value, str):
                    # Escape single quotes for Cypher
                    escaped = value.replace("'", "\\'")
                    cleaned_data[key] = f"'{escaped}'"
                elif isinstance(value, list):
                    # Convert list to string array
                    str_list = []
                    for v in value:
                        escaped_v = str(v).replace("'", "\\'")
                        str_list.append(f"'{escaped_v}'")
                    cleaned_data[key] = f"[{','.join(str_list)}]"
                else:
                    cleaned_data[key] = str(value)

            # Build CREATE statement
            fields = ", ".join([f"{k}: {v}" for k, v in cleaned_data.items()])
            create_sql = f"CREATE (:{node_type} {{{fields}}})"
            try:
                conn.execute(create_sql)
            except Exception as e:
                print(f"Error creating {node_type} node: {e}")
                print(f"SQL: {create_sql}")

    # Load relationships
    print(f"Loading {len(relationships)} relationships...")
    for from_type, from_key, to_type, to_key, rel_name in relationships:
        # Escape keys
        from_key_escaped = from_key.replace("'", "\\'")
        to_key_escaped = to_key.replace("'", "\\'")

        # Find node keys
        from_node_key = next(n.key for n in nodes if n.baml_class.__name__ == from_type)
        to_node_key = next(n.key for n in nodes if n.baml_class.__name__ == to_type)

        match_sql = f"""
        MATCH (from:{from_type}), (to:{to_type})
        WHERE from.{from_node_key} = '{from_key_escaped}'
          AND to.{to_node_key} = '{to_key_escaped}'
        CREATE (from)-[:{rel_name}]->(to)
        """

        try:
            conn.execute(match_sql)
        except Exception as e:
            print(f"Error creating {rel_name} relationship: {e}")
            print(f"SQL: {match_sql}")


def create_graph(model: BaseModel, nodes: list[NodeInfo], relations: list[RelationInfo]) -> None:
    """Create a knowledge graph from a Pydantic model in Kuzu database.

    Args:
        model: ReviewedOpportunity instance to convert
        nodes: List of node type definitions
        relations: List of relationship definitions
    """
    print("Creating database schema...")
    create_schema(conn, nodes, relations)

    print("Extracting graph data...")
    nodes_dict, relationships = extract_graph_data(model, nodes, relations)

    print("Loading data into graph...")
    load_graph_data(conn, nodes_dict, relationships, nodes)

    # Print statistics
    total_nodes = sum(len(node_list) for node_list in nodes_dict.values())
    print("\nGraph creation complete!")
    print(f"Total nodes: {total_nodes}")
    print(f"Total relationships: {len(relationships)}")

In [ ]:
# Create the knowledge graph from the CNES opportunity data
# Note: If you get 'Person already exists in catalog' error, run: restart_database()

assert opportunity
create_graph(opportunity, nodes, relations)

In [ ]:
# Validation queries to check the created graph

print("=== Graph Statistics ===")

# Count nodes by type
for node_info in nodes:
    node_type = node_info.baml_class.__name__
    try:
        result = conn.execute(f"MATCH (n:{node_type}) RETURN count(n) as count")
        count = result.get_as_df().iloc[0]["count"]
        print(f"{node_type}: {count} nodes")
    except Exception as e:
        print(f"Error counting {node_type}: {e}")

# Count relationships by type
print("\n=== Relationships ===")
for relation in relations:
    rel_name = relation.name
    try:
        result = conn.execute(f"MATCH ()-[r:{rel_name}]->() RETURN count(r) as count")
        count = result.get_as_df().iloc[0]["count"]
        print(f"{rel_name}: {count} relationships")
    except Exception as e:
        print(f"Error counting {rel_name}: {e}")

# Sample queries
print("\n=== Sample Data ===")

# Show opportunity details
print("Opportunity Details:")
result = conn.execute("MATCH (o:Opportunity) RETURN o.name as name, o.opportunity_id as id, o.status as status")
print(result.get_as_df())

# Show customer and contacts
print("\nCustomer and Contacts:")
result = conn.execute("""
    MATCH (o:Opportunity)-[:HAS_CUSTOMER]->(c:Customer)-[:HAS_CONTACT]->(p:Person) 
    RETURN c.name as customer, p.name as contact, p.role as role
""")
print(result.get_as_df())

# Show team members
print("\nTeam Members:")
result = conn.execute("""
    MATCH (o:Opportunity)-[:HAS_TEAM_MEMBER]->(p:Person) 
    RETURN p.name as name, p.role as role, p.organization as org
    LIMIT 5
""")
print(result.get_as_df())

# Show risks
print("\nRisks:")
result = conn.execute("""
    MATCH (o:Opportunity)-[:HAS_RISK]->(r:RiskAnalysis) 
    RETURN r.risk_description as description, r.impact_level as impact, r.status as status
    LIMIT 3
""")
print(result.get_as_df())

# Show financial metrics
print("\nFinancials:")
result = conn.execute("""
    MATCH (o:Opportunity)-[:HAS_FINANCIALS]->(f:FinancialMetrics) 
    RETURN f.tcv as tcv, f.annual_revenue as revenue, f.project_margin as margin
""")
print(result.get_as_df())

In [ ]:
def get_cytoscape_json(conn) -> dict:
    """Convert GraphStore data to Cytoscape JSON format.

    Args:
        graph: The GraphStore containing nodes and relationships

    Returns:
        dict: Cytoscape-compatible JSON with nodes and edges, excluding text_chunk nodes
    """
    cyto_data = {"nodes": [], "edges": []}

    # Query all nodes
    nodes_query = "MATCH (n) RETURN n"
    nodes_result = conn.execute(nodes_query)

    # Add nodes to Cytoscape data, excluding text_chunk nodes
    for node in nodes_result:
        if node["n"]["type"] != "text_chunk":
            cyto_data["nodes"].append(
                {"data": {"id": node["n"]["id"], "label": node["n"]["_label"], "type": node["n"]["type"]}}
            )

    # Query all relationships
    rels_query = "MATCH (a)-[r]->(b) RETURN a, r, b"
    rels_result = conn.execute(rels_query)

    # Add relationships to Cytoscape data, excluding those involving text_chunk nodes
    for rel in rels_result:
        if rel["a"]["type"] != "text_chunk" and rel["b"]["type"] != "text_chunk":
            cyto_data["edges"].append(
                {
                    "data": {
                        "id": f"{rel['a']['id']}-{rel['r']['_label']}-{rel['b']['id']}",
                        "source": rel["a"]["id"],
                        "target": rel["b"]["id"],
                        "label": rel["r"]["_label"],
                    }
                }
            )
    return cyto_data


def get_cytoscape_style() -> list[dict]:
    """Get default styling for Cytoscape graph visualization.

    Returns:
        list[dict]: List of style definitions for nodes and edges
    """
    style = [
        {
            "selector": "node",
            "css": {
                "content": "data(id)",
                "text-valign": "center",
                "text-halign": "center",
                "color": "white",
                "background-color": "#002733",
                "width": "label",
                "height": "label",
                "padding": "10px",
                "shape": "ellipse",
                "font-size": "12px",
                "border-width": 2,
                "border-color": "#444",
            },
        },
        {
            "selector": "edge",
            "css": {
                "content": "data(label)",
                "curve-style": "bezier",
                "target-arrow-shape": "triangle",
                "line-color": "#999",
                "target-arrow-color": "#999",
                "font-size": "6px",
                "text-margin-y": "-10px",
                "text-rotation": "autorotate",
                "width": 2,
                "arrow-scale": 1,
            },
        },
    ]
    return style


In [ ]:
from ipycytoscape import CytoscapeWidget

cyto = CytoscapeWidget()
graph_json = get_cytoscape_json(conn)
cyto.graph.add_graph_from_json(graph_json)
cyto.set_style(get_cytoscape_style())
# Set layout and style
cyto.set_layout(animate=True)